In [8]:
import pyreadr

# Load the rds file
result = pyreadr.read_r('06_Cb_BTM_table.rds')

# pyreadr returns a dictionary; the data is usually under the 'None' key 
# because rds files contain a single object without a specific name.
df = result[None]

# select ttype 

ttype = "OV"

df = df[df["ttype"]==ttype]

df = df.dropna(subset=["clock_rank","gene"])

trees = {}

patients = df["sample_id"].unique()

interesting_p = 0

for p in patients:
    df_p = df[df["sample_id"] == p]
    ranks = int(df_p["clock_rank"].unique().max())
    genes = {}
    for r in range(1,ranks+1):
        df_p_r =  df_p[df_p["clock_rank"]==r]
        genes_list = df_p_r["gene"].tolist()
        if len(genes_list)>0: #some ranks are skipped ????
            genes[r] = df_p_r["gene"].tolist()
    trees[p] = genes
    interesting_p += 1
    

In [27]:
# tring some mixup
from collections import defaultdict

# map gene end in which clock was found with multeplicity

r_gen_all = defaultdict(lambda: defaultdict(int))

for p, ranks in trees.items():
    for r, genes_list in ranks.items():
        for g in genes_list:
            r_gen_all[g][r] += 1
            
print("All gens found : ",len(r_gen_all))
            
            
# drop genes that appear only one time

print("Number of patients ",len(patients))
mult_treshold = len(patients)/3
print("setting treshold to ",mult_treshold)
temp = {}
for k in r_gen_all.keys():
    gen_compa = r_gen_all[k]
    if sum(gen_compa.values()) > mult_treshold:
        temp[k] = r_gen_all[k]
r_gen_all = temp

print("genes with multeplicity higher than ",mult_treshold," : ",len(r_gen_all))


# prob where to place items
r_gen_refined = {}
for k,v in r_gen_all.items():
    r_gen_refined[k] = max(v, key=v.get)

# create tree
multi_tree = defaultdict(list)
for k,v in r_gen_refined.items():
    multi_tree[v].append(k)

All gens found :  591
Number of patients  66
setting treshold to  22.0
genes with multeplicity higher than  22.0  :  99
